# Cognopolis · M1 — Воркспейс: собираем простейшего агента

Это **воркспейс** урока [M1 «Реактивный житель-сборщик»](https://itrubnikov.github.io/Train_of_Thought/docs/game-lessons/m1-reactive-gatherer/) —
разбираем его вместе на занятии (или самостоятельно, сверху вниз). Здесь всё уже собрано и
работает; твоя собственная сборка — в домашке (`notebook.ipynb` в этой же папке).

Собираем самого простого агента **четырьмя слоями**, каждый поверх предыдущего:

1. **Голый REST** — «глаза» и «руки» агента: два GET и один POST.
2. **Инструменты (tools)** — те же вызовы, завёрнутые в функции с понятными описаниями.
3. **Реактивный цикл** — простейший агент: `observe -> decide -> act -> wait`, «мозг» = пара правил.
4. **Фреймворк (smolagents)** — тот же агент, но решения принимает LLM. Превью урока M4.

Ключевая мысль: **агент — это цикл**. Слои 3 и 4 используют одни и те же инструменты
слоя 2; меняется только то, кто решает — твой код или модель.


In [ ]:
%pip install -q requests "smolagents[openai]"

In [ ]:
import os

def read_secret(name: str, default: str = "") -> str:
    """Секрет из Colab userdata -> Kaggle Secrets -> переменной окружения -> default."""
    try:
        from google.colab import userdata          # Colab: Secrets на панели слева
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle: Add-ons -> Secrets
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)

# ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Свой сервер: секрет/env COGNOPOLIS_URL.
BASE_URL = read_secret("COGNOPOLIS_URL", "https://kindomklaster.com").rstrip("/")

# ТВОЙ ТОКЕН (полный доступ к твоему жителю):
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша -> вкладка «аккаунт» -> «копировать» — это твой токен.
#   3. Положи его в секрет COGNOPOLIS_TOKEN (Colab/Kaggle) или в переменную окружения.
TOKEN = read_secret("COGNOPOLIS_TOKEN")
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

print("Мир:", BASE_URL)
print("Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN[:6]}...  (полная ссылка = BASE_URL/?token=<твой токен>)")


## Слой 1 — голый REST: глаза и руки

У агента нет «прямого зрения»: всё, что он знает о мире, приходит из HTTP-ответов.
Весь API описан на `BASE_URL/docs` (кнопка **Authorize** — вставь токен один раз).

- `GET /character` — где я, hp, рюкзак, склад и **cooldown** (сколько секунд я ещё «занят»);
- `GET /map` — карта: клетки с ресурсами (`tree`, `rock`) и враги;
- `POST /actions/move/<направление>` — шаг на одну клетку (8 сторон света, D-069);
- `POST /actions/gather` — добыть ресурс, **стоя на его клетке**.

Каждое действие возвращает `{result, cooldown, character}`; нарушение правил — ошибку
`{"error": {"code", "message"}}` с честным кодом (`no_resource_here`, `inventory_full`...).


In [ ]:
import time

import requests

me = requests.get(f"{BASE_URL}/character", headers=HEADERS, timeout=15).json()
print("я в", (me["x"], me["y"]), "| hp", me["hp"], "/", me["max_hp"], "| рюкзак", me["inventory"])

world = requests.get(f"{BASE_URL}/map", timeout=15).json()   # карта публична, токен не нужен
print("карта", world["size"], "×", world["size"], "| на клетках:", sorted({t["content"] for t in world["tiles"]}))

time.sleep(me["cooldown"])                                   # житель мог быть «занят» после прошлых прогонов

direction = "east" if me["x"] < world["size"] - 1 else "west"    # не упереться в край карты
r = requests.post(f"{BASE_URL}/actions/move/{direction}", headers=HEADERS,
                  json={"reason": "разогрев: пробую сходить"}, timeout=15).json()
if "error" in r:
    print("сервер отбил действие:", r["error"]["code"], "—", r["error"]["message"])
else:
    print(f"шаг {direction}: теперь", (r["character"]["x"], r["character"]["y"]), "| cooldown", r["cooldown"], "с")
    time.sleep(r["cooldown"])                                # wait: кулдаун — естественный ритм петли


## Слой 2 — инструменты: те же вызовы, но с контрактом

Оборачиваем REST в четыре функции — будущие tools агента. Три правила хорошего
инструмента (их разбирает модуль 11.5 курса):

- **описание объясняет, когда инструмент нужен** — как для нового стажёра;
- **ошибка учит чиниться**: не исключение-трейсбек, а понятная строка с кодом
  (`не вышло: no_resource_here` подсказывает — сначала встань на клетку ресурса);
- **инструмент сам выдерживает кулдаун** — каким бы «мозгом» ни крутился цикл,
  спамить сервер физически невозможно.


In [ ]:
HOME = (0, 0)          # дом: шаг на эту клетку авто-разгружает рюкзак на склад
DIRECTIONS = ("north", "south", "east", "west", "northeast", "northwest", "southeast", "southwest")


def api_get(path: str) -> dict:
    """GET к игре (глаза агента): /character, /map, /assignment."""
    r = requests.get(BASE_URL + path, headers=HEADERS, timeout=15)
    data = r.json()
    if not r.ok:
        raise RuntimeError((data.get("error") or {}).get("code", str(r.status_code)))
    return data


def act(path: str, **body):
    """POST-действие (руки агента) -> (result, None) или (None, "код ошибки").
    Сам выдерживает кулдаун после успеха и делает паузу после отказа."""
    r = requests.post(BASE_URL + path, headers=HEADERS, json=body or None, timeout=15)
    data = r.json()
    if not r.ok:
        time.sleep(1)                                   # отказ не двигает кулдаун — но не молотим сервер
        return None, (data.get("error") or {}).get("code", str(r.status_code))
    time.sleep(float(data.get("cooldown", 0)))          # wait встроен в каждое действие
    return data, None


def get_status() -> str:
    """Где житель и что у него с собой: позиция, hp, рюкзак (с ёмкостью), склад."""
    ch = api_get("/character")
    carried = sum(ch["inventory"].values())
    return (f"позиция ({ch['x']},{ch['y']}); hp {ch['hp']}/{ch['max_hp']}; "
            f"рюкзак {ch['inventory']} ({carried}/{ch['inventory_cap']}); "
            f"склад {ch['stored']}; дом {HOME}")


def look_around() -> str:
    """Что где на карте: ресурсы (и враги — М1 их обходит) с координатами и дистанцией."""
    ch, world = api_get("/character"), api_get("/map")
    lines = [f"карта {world['size']}x{world['size']}; я в ({ch['x']},{ch['y']}); дом {HOME}"]
    for t in world["tiles"]:
        if t["content"] in ("tree", "rock"):
            d = abs(t["x"] - ch["x"]) + abs(t["y"] - ch["y"])
            lines.append(f"- {t['content']} в ({t['x']},{t['y']}), дистанция {d}")
    for e in world.get("enemies", []):
        if e["alive"]:
            lines.append(f"- враг {e['kind']} в ({e['x']},{e['y']}) — в M1 обходим стороной")
    return "\n".join(lines)


def move(direction: str) -> str:
    """Шаг на одну клетку в направлении из DIRECTIONS (8 сторон света)."""
    if direction not in DIRECTIONS:
        return f"не вышло: unknown_direction (можно: {', '.join(DIRECTIONS)})"
    res, err = act(f"/actions/move/{direction}", reason=f"шаг {direction}")
    if err:
        return f"не вышло: {err}"
    ch = res["character"]
    out = f"шаг {direction}: теперь ({ch['x']},{ch['y']})"
    if res["result"].get("banked"):
        out += f" · дом принял на склад: {res['result']['banked']}"
    return out


def gather() -> str:
    """Добыть ресурс на СВОЕЙ клетке (сначала встань на tree/rock шагами move)."""
    res, err = act("/actions/gather", reason="добываю")
    if err:
        return f"не вышло: {err}"
    r, ch = res["result"], res["character"]
    carried = sum(ch["inventory"].values())
    return f"добыл {r['amount']}x{r['gathered']} (рюкзак {carried}/{ch['inventory_cap']})"


start = api_get("/character")
WOOD_AT_START = start["inventory"].get("wood", 0) + start["stored"].get("wood", 0)

print(get_status())
print(look_around())


## Слой 3 — простейший агент: реактивный цикл

«Мозг» — два правила, решение каждый ход рождается заново из текущего состояния:

- рюкзак **полон** -> идти домой (дом сам принимает добычу на склад);
- иначе -> идти к ближайшему дереву и **добывать, стоя на нём**.

Никакого маршрута никто не прокладывал — поведение целиком из условий на состояние.
Обрати внимание: `act`-часть — это те же `move()` и `gather()` из слоя 2.


In [ ]:
STEP_DIR = {(1, 0): "east", (-1, 0): "west", (0, 1): "south", (0, -1): "north",
            (1, 1): "southeast", (1, -1): "northeast", (-1, 1): "southwest", (-1, -1): "northwest"}


def one_reactive_step() -> str:
    """Один ход простейшего агента: observe -> decide -> act (wait встроен в действия)."""
    ch = api_get("/character")                                  # observe
    world = api_get("/map")
    carried = sum(ch["inventory"].values())
    if carried >= ch["inventory_cap"]:                          # decide: полон -> домой
        tx, ty = HOME
    else:                                                       # decide: к ближайшему дереву
        trees = [t for t in world["tiles"] if t["content"] == "tree"]
        node = min(trees, key=lambda t: abs(t["x"] - ch["x"]) + abs(t["y"] - ch["y"]))
        tx, ty = node["x"], node["y"]
    if (ch["x"], ch["y"]) == (tx, ty):                          # act: на месте — добываем/ждём
        return gather() if (tx, ty) != HOME else "дома: рюкзак разгружен на заходе"
    dx = (tx > ch["x"]) - (tx < ch["x"])
    dy = (ty > ch["y"]) - (ty < ch["y"])
    return move(STEP_DIR[(dx, dy)])                             # act: шаг к цели


time.sleep(api_get("/character")["cooldown"])                   # дождаться хвоста кулдауна
for step in range(12):
    print(f"[{step:2}]", one_reactive_step())


## Слой 4 — тот же агент на фреймворке (smolagents)

Теперь вместо наших `if` решение принимает **LLM**: мы отдаём ей те же инструменты,
обёрнутые в `@tool`, и текстовую задачу. Это превью урока M4 — там LLM tool-use
разбирается всерьёз; здесь важно увидеть: **инструменты не изменились, поменялся только
«мозг» цикла**.

Модель подключается любая OpenAI-совместимая, по порядку:

- **локальная** — LM Studio (`http://localhost:1234/v1`) или Ollama (`http://localhost:11434/v1`):
  задай секрет/env `LOCAL_LLM_URL` (+ `LOCAL_LLM_MODEL` — имя модели, например `qwen2.5:7b`);
- **MiniMax** (канон курса) — секрет `MINIMAX_API_KEY`;
- ничего нет — ячейка честно пропустит слой (реактивный агент выше уже всё умеет).


In [ ]:
from smolagents import OpenAIServerModel, ToolCallingAgent, tool


@tool
def game_status() -> str:
    """Сообщает, где сейчас житель и что у него с собой.

    Returns:
        Позиция, hp, рюкзак с ёмкостью, склад и координаты дома.
    """
    return get_status()


@tool
def game_look() -> str:
    """Осматривает карту: какие ресурсы где лежат и как далеко до них.

    Returns:
        Список ресурсов (tree/rock) и врагов с координатами и дистанцией.
    """
    return look_around()


@tool
def game_move(direction: str) -> str:
    """Делает один шаг в указанном направлении.

    Args:
        direction: одно из north/south/east/west/northeast/northwest/southeast/southwest.

    Returns:
        Новая позиция жителя, либо строка "не вышло: <код>" с причиной отказа.
    """
    return move(direction)


@tool
def game_gather() -> str:
    """Добывает ресурс на клетке, где житель стоит прямо сейчас.

    Returns:
        Что добыто и насколько заполнен рюкзак, либо "не вышло: <код>"
        (например no_resource_here — сначала дойди до клетки с ресурсом).
    """
    return gather()


LOCAL_LLM_URL = read_secret("LOCAL_LLM_URL")
MINIMAX_API_KEY = read_secret("MINIMAX_API_KEY")


def make_model():
    """Локальная OpenAI-совместимая модель -> MiniMax -> None (слой пропускается)."""
    if LOCAL_LLM_URL:
        return OpenAIServerModel(model_id=read_secret("LOCAL_LLM_MODEL", "local-model"),
                                 api_base=LOCAL_LLM_URL.rstrip("/"),
                                 api_key=read_secret("LOCAL_LLM_KEY", "no-key"))
    if MINIMAX_API_KEY:
        return OpenAIServerModel(model_id="MiniMax-M3", api_base="https://api.minimax.io/v1",
                                 api_key=MINIMAX_API_KEY)
    return None


TASK = (
    "Ты управляешь жителем игры Cognopolis. Сначала осмотрись (game_look) и проверь состояние "
    "(game_status). Затем собери 2 единицы дерева: дойди шагами game_move до клетки с tree "
    "(добыча работает только СТОЯ на клетке ресурса) и дважды успешно вызови game_gather. "
    "После этого вернись домой в клетку (0,0). В конце коротко отчитайся, что собрал и где стоишь."
)

model = make_model()
if model is None:
    print("LLM не настроен — слой 4 пропущен (это нормально для M1: реактивный агент выше уже всё умеет).")
    print("Включить: секрет LOCAL_LLM_URL (LM Studio/Ollama) или MINIMAX_API_KEY (MiniMax).")
else:
    agent = ToolCallingAgent(tools=[game_status, game_look, game_move, game_gather],
                             model=model, max_steps=12)
    print("Ответ агента:", agent.run(TASK))


## Проверка

Оба прогона (реактивный цикл и, если была модель, smolagents) работали в одном общем
мире — так что просто смотрим суммарный прирост дерева с начала воркспейса.


In [ ]:
ch = api_get("/character")
wood_now = ch["inventory"].get("wood", 0) + ch["stored"].get("wood", 0)
print(f"дерева было {WOOD_AT_START} -> стало {wood_now} (рюкзак {ch['inventory']}, склад {ch['stored']})")

assert wood_now >= WOOD_AT_START + 3, "Цель воркспейса: +3 дерева. Перезапусти слой 3 (или проверь, что мир доступен)."
print("Воркспейс пройден: простейший агент собран и работает.")


## Наблюдаемость и что дальше

Открой `BASE_URL/?token=<твой токен>` во второй вкладке: там видно шаги жителя, его
«мысль» (`reason` каждого действия) и Хронику. Один и тот же агент — в ноутбуке цикл,
в браузере живой житель. «1 житель = 1 агент».

**Домашка** — `notebook.ipynb` в этой же папке: собери такого же агента сам. Каркас уже
рабочий; твои задачи — разгрузка при полном рюкзаке, баланс дерево/камень и чтение
поручения из Ратуши (плюс задача со звёздочкой — свой агент на smolagents, как в слое 4).

Дальше по курсу: [M2 — боевой цикл с отступлением](https://itrubnikov.github.io/Train_of_Thought/docs/game-lessons/m2-combat/),
а LLM-«мозг» всерьёз — в M4.
